# Session 6 - RAG Fundamentals

AI Agents & Workflows: Basics

In this notebook we will build a small RAG application step by step.

We will cover:

- what RAG is
- why plain LLM calls are not enough
- documents and metadata
- text splitting
- embeddings
- vector stores
- similarity search
- retrievers
- building a simple RAG chain
- returning answers with sources

Main mental model:

```text
RAG = Retrieve relevant context + Generate grounded answer
```

## 0. Setup

This notebook expects:

- `.env` file in the project root
- `OPENAI_API_KEY` inside `.env`
- optional `OPENAI_MODEL` inside `.env`
- optional `OPENAI_EMBEDDING_MODEL` inside `.env`

Example:

```bash
OPENAI_API_KEY=your_api_key_here
OPENAI_MODEL=gpt-4o-mini
OPENAI_EMBEDDING_MODEL=text-embedding-3-small
```

Required packages:

```bash
pip install langchain langchain-openai langchain-text-splitters python-dotenv
```

In [ ]:
import os
from dotenv import load_dotenv

# The notebook is inside notebooks/, so we load ../.env from the project root.
load_dotenv("../.env")

api_key = os.getenv("OPENAI_API_KEY")
model_name = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
embedding_model_name = os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small")

print("OPENAI_API_KEY loaded:", bool(api_key))
print("Chat model:", model_name)
print("Embedding model:", embedding_model_name)

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model=model_name,
    temperature=0
)

print("LLM initialized")

## 1. Why RAG?

A plain LLM call can answer general questions, but it does not automatically know our internal documentation.

Example:

```text
How do we onboard a new Kubernetes namespace in our platform?
```

Without context, the model can only guess.

In [ ]:
question = "How do we onboard a new Kubernetes namespace in our internal platform?"

response = llm.invoke(question)

print(response.content)

### Discussion

The answer may sound reasonable, but it is not grounded in our actual internal documents.

For internal systems, we need to provide the right context.

That is what RAG does.

## 2. Create a Small Knowledge Base

For the demo, we will create a small in-memory knowledge base.

In a real project, these documents could come from:

- Markdown files
- PDFs
- Confluence
- Jira
- Git repositories
- runbooks
- internal APIs

In [ ]:
from langchain_core.documents import Document

docs = [
    Document(
        page_content=(
            "Kubernetes namespace onboarding requires a namespace name, owner team, "
            "environment type, resource quota, network policy, and required labels. "
            "The request should be reviewed by the platform team before creation."
        ),
        metadata={"source": "namespace_onboarding.md", "section": "requirements"}
    ),
    Document(
        page_content=(
            "For production namespaces, the requester must provide monitoring contacts, "
            "alert routing information, backup requirements, and escalation owner. "
            "Production namespaces require approval from the platform owner."
        ),
        metadata={"source": "namespace_onboarding.md", "section": "production"}
    ),
    Document(
        page_content=(
            "Workload troubleshooting starts by checking pod status, recent events, "
            "container logs, readiness probes, liveness probes, service endpoints, "
            "and recent deployment changes."
        ),
        metadata={"source": "workload_troubleshooting.md", "section": "checklist"}
    ),
    Document(
        page_content=(
            "Argo CD applications should define source repository, target revision, "
            "destination namespace, sync policy, health status, and rollback procedure. "
            "Automated sync should be used carefully for production workloads."
        ),
        metadata={"source": "argocd_deployment_model.md", "section": "application"}
    ),
    Document(
        page_content=(
            "Storage requests must specify storage class, access mode, requested size, "
            "retention expectation, backup requirements, and workload mapping. "
            "PVC usage should be reviewed during operational readiness checks."
        ),
        metadata={"source": "storage_review.md", "section": "pvc"}
    ),
]

print(f"Documents: {len(docs)}")

for doc in docs:
    print(doc.metadata, "=>", doc.page_content[:80] + "...")

## 3. Baseline: Simple Keyword Search

Before vector search, let's build a very simple keyword retriever.

This is not real RAG yet, but it helps us understand the retrieval problem.

In [ ]:
def keyword_search(query: str, documents: list[Document], top_k: int = 2):
    query_words = set(query.lower().split())
    scored = []

    for doc in documents:
        doc_words = set(doc.page_content.lower().split())
        score = len(query_words.intersection(doc_words))
        scored.append((score, doc))

    scored.sort(key=lambda item: item[0], reverse=True)
    return [doc for score, doc in scored[:top_k] if score > 0]


results = keyword_search("production namespace approval", docs)

for doc in results:
    print(doc.metadata)
    print(doc.page_content)
    print("---")

### Keyword Search Limitation

Keyword search depends on exact words.

If the question uses different wording, keyword search may miss relevant documents.

RAG usually uses embeddings for semantic search.

## 4. Text Splitting

Large documents should be split into smaller chunks.

Why?

- better retrieval quality
- lower context usage
- more focused answers
- easier source tracking

For this demo our documents are short, but we still show the pattern.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=180,
    chunk_overlap=30
)

splits = text_splitter.split_documents(docs)

print(f"Original documents: {len(docs)}")
print(f"Chunks: {len(splits)}")

for i, chunk in enumerate(splits[:5], start=1):
    print(f"Chunk {i}")
    print(chunk.metadata)
    print(chunk.page_content)
    print("---")

## 5. Embeddings

Embeddings convert text into vectors.

```text
Text → Embedding model → Vector of numbers
```

Texts with similar meaning should have similar vectors.

In [ ]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model=embedding_model_name)

sample_vector = embeddings.embed_query("Kubernetes namespace onboarding")

print(type(sample_vector))
print("Vector length:", len(sample_vector))
print("First 5 values:", sample_vector[:5])

## 6. Vector Store

A vector store stores:

- document chunks
- embeddings
- metadata

For this demo we will use an in-memory vector store.

For production you may use Chroma, Qdrant, Pinecone, Weaviate, FAISS, or PostgreSQL + pgvector.

In [ ]:
from langchain_core.vectorstores import InMemoryVectorStore

vector_store = InMemoryVectorStore(embeddings)

document_ids = vector_store.add_documents(documents=splits)

print(f"Added documents: {len(document_ids)}")

## 7. Similarity Search

Now we can search by semantic meaning.

The query is embedded and compared with stored document vectors.

In [ ]:
query = "What approval is needed for a production namespace?"

similar_docs = vector_store.similarity_search(query, k=3)

for i, doc in enumerate(similar_docs, start=1):
    print(f"Result {i}")
    print("Metadata:", doc.metadata)
    print("Content:", doc.page_content)
    print("---")

## 8. Retriever

A retriever is a standard interface for getting relevant documents.

It hides the search implementation behind a simple `.invoke()` call.

In [ ]:
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

retrieved_docs = retriever.invoke("How do I troubleshoot a failing workload?")

for doc in retrieved_docs:
    print(doc.metadata)
    print(doc.page_content)
    print("---")

## 9. Format Retrieved Documents

Before sending documents to the LLM, we usually format them as context.

In [ ]:
def format_docs(documents: list[Document]) -> str:
    formatted = []

    for i, doc in enumerate(documents, start=1):
        source = doc.metadata.get("source", "unknown")
        section = doc.metadata.get("section", "unknown")
        formatted.append(
            f"[Source {i}: {source} / {section}]\n{doc.page_content}"
        )

    return "\n\n".join(formatted)


context = format_docs(retrieved_docs)
print(context)

## 10. Basic RAG Prompt

A good RAG prompt should tell the model:

- use the provided context
- do not invent missing information
- answer the question
- mention sources when useful

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

rag_prompt = ChatPromptTemplate.from_messages([
    ("system", """
You are a helpful internal engineering assistant.

Answer the question using only the provided context.

If the context does not contain the answer, say:
"I do not have enough information in the provided context."

Keep the answer concise and practical.
"""),
    ("human", """
Context:
{context}

Question:
{question}
""")
])

rag_chain = rag_prompt | llm | StrOutputParser()

print("RAG prompt created")

In [ ]:
question = "How do I troubleshoot a failing Kubernetes workload?"

retrieved_docs = retriever.invoke(question)
context = format_docs(retrieved_docs)

answer = rag_chain.invoke({
    "context": context,
    "question": question
})

print(answer)

## 11. RAG Function

Now let's wrap the full RAG flow in a function.

```text
question
  ↓
retriever
  ↓
context
  ↓
prompt
  ↓
llm
  ↓
answer
```

In [ ]:
def ask_rag(question: str, k: int = 3) -> dict:
    docs_for_question = vector_store.similarity_search(question, k=k)
    context = format_docs(docs_for_question)

    answer = rag_chain.invoke({
        "context": context,
        "question": question
    })

    return {
        "question": question,
        "answer": answer,
        "sources": docs_for_question,
    }

In [ ]:
result = ask_rag("What information is required for namespace onboarding?")

print("Answer:")
print(result["answer"])

print("\nSources:")
for doc in result["sources"]:
    print("-", doc.metadata)

## 12. RAG With Sources

For internal assistants, sources are very important.

They help users verify the answer.

In [ ]:
def ask_rag_with_sources(question: str, k: int = 3) -> str:
    result = ask_rag(question, k=k)

    source_lines = []
    seen = set()

    for doc in result["sources"]:
        source = doc.metadata.get("source", "unknown")
        section = doc.metadata.get("section", "unknown")
        key = (source, section)

        if key not in seen:
            seen.add(key)
            source_lines.append(f"- {source} / {section}")

    return f"""{result["answer"]}

Sources:
{chr(10).join(source_lines)}
"""

print(ask_rag_with_sources("What is required for production namespaces?"))

## 13. Unknown Answer Example

A good RAG system should not answer when the context does not contain the information.

Let's ask something outside our knowledge base.

In [ ]:
print(ask_rag_with_sources("What is the company's official travel reimbursement policy?"))

## 14. Compare: No Context vs RAG Context

This is a useful demo for students.

Same question, different context availability.

In [ ]:
question = "What approvals are needed for production namespaces?"

no_context_answer = llm.invoke(question).content
rag_answer = ask_rag_with_sources(question)

print("NO CONTEXT ANSWER:")
print(no_context_answer)

print("\n" + "="*80 + "\n")

print("RAG ANSWER:")
print(rag_answer)

## 15. RAG vs Memory

Memory and RAG are different:

```text
Memory = previous conversation context

RAG = external knowledge retrieval
```

They can work together.

Example:

- Memory remembers that the user is working on Kubernetes onboarding.
- RAG retrieves the onboarding template.

## 16. RAG vs Agent

RAG is usually a retrieval pipeline.

An agent can decide whether to use RAG.

```text
User question
  ↓
Agent decides:
    - answer directly
    - retrieve docs
    - call tool
    - ask clarification
```

RAG can be a tool inside an agent.

## 17. Simple RAG as a Tool

Let's wrap RAG as a function that an agent could use.

In [ ]:
from langchain_core.tools import tool

@tool
def search_internal_docs(query: str) -> str:
    """Search internal engineering documentation and return relevant context."""
    docs_for_query = vector_store.similarity_search(query, k=3)
    return format_docs(docs_for_query)


print(search_internal_docs.invoke("namespace production approval"))

## 18. Optional: Agent Using RAG Tool

Now the agent can decide when to use the RAG tool.

This combines Session 3 agents with Session 6 RAG.

In [ ]:
from langchain.agents import create_agent

rag_agent = create_agent(
    model=llm,
    tools=[search_internal_docs],
    system_prompt=(
        "You are an internal engineering assistant. "
        "Use the documentation search tool when the user asks about internal engineering procedures. "
        "If the docs do not contain the answer, say that you do not have enough information."
    )
)

print("RAG agent created")

In [ ]:
agent_result = rag_agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "What do we need for production namespace onboarding?"
        }
    ]
})

print(agent_result["messages"][-1].content)

## 19. Metadata Filtering Concept

In production, metadata can be used to filter retrieval.

Examples:

```text
team = platform
environment = production
source_type = runbook
user_has_access = true
updated_after = 2026-01-01
```

This is important for security and relevance.

In [ ]:
# Simple manual metadata filtering example
platform_docs = [
    doc for doc in splits
    if doc.metadata.get("source") in ["namespace_onboarding.md", "workload_troubleshooting.md"]
]

print(f"Filtered chunks: {len(platform_docs)}")

for doc in platform_docs:
    print(doc.metadata)

## 20. Common RAG Problems

RAG can fail in several places:

1. Bad data source
2. Poor chunking
3. Wrong embedding model
4. Bad retrieval
5. Too many irrelevant chunks
6. Missing metadata
7. Stale documents
8. LLM ignores context
9. No source citations
10. No evaluation

## 21. Practical RAG Checklist

When building RAG, check:

- Are the documents clean?
- Are chunks the right size?
- Are sources stored in metadata?
- Does retrieval return the right chunks?
- Does the prompt prevent guessing?
- Are sources shown to the user?
- Is access control respected?
- Is there an evaluation set?

## 22. Exercises

### Exercise 1

Add two new documents to the knowledge base:

- one about Jenkins pipelines
- one about monitoring / Grafana

Rebuild the vector store and test retrieval.

### Exercise 2

Change `k` from 3 to 1 and 5.

Compare answer quality.

### Exercise 3

Change chunk size and chunk overlap.

Observe how retrieval changes.

### Exercise 4

Add a source citation format to the final answer.

## 23. Discussion Questions

1. What problem does RAG solve?
2. Why do we split documents into chunks?
3. What is an embedding?
4. What is a vector store?
5. Why is metadata important?
6. What is the difference between RAG and memory?
7. How can an agent use RAG?
8. What can go wrong in a RAG system?

## 24. Key Takeaways

- RAG means Retrieval-Augmented Generation.
- RAG gives LLMs access to external knowledge.
- Documents are split into chunks.
- Embeddings convert text into vectors.
- Vector stores enable semantic similarity search.
- Retrieved context is injected into the prompt.
- RAG should answer from context, not guess.
- Sources are important for trust.
- Agents can use RAG as a tool.

## 25. Official Documentation

- Retrieval: https://docs.langchain.com/oss/python/langchain/retrieval
- Build a semantic search engine: https://docs.langchain.com/oss/python/langchain/knowledge-base
- Embeddings: https://docs.langchain.com/oss/python/integrations/embeddings
- Vector stores: https://docs.langchain.com/oss/python/integrations/vectorstores
- OpenAI embeddings: https://docs.langchain.com/oss/python/integrations/embeddings/openai